[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/multiple-schemas.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239434-lesson-3-multiple-schemas)

# 多重模式

## 回顾

我们刚刚学习了状态模式和reducer。

通常，所有图节点都与单个模式通信。

此外，这个单一模式包含图的输入和输出键/通道。

## 目标

但是，在某些情况下，我们可能希望对此有更多的控制：

* 内部节点可能传递在图的输入/输出中*不需要*的信息。

* 我们还可能希望为图使用不同的输入/输出模式。例如，输出可能只包含单个相关的输出键。

我们将讨论使用多重模式自定义图的几种方法。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph

## 私有状态

首先，让我们讨论在节点之间传递[私有状态](https://langchain-ai.github.io/langgraph/how-tos/pass_private_state/)的情况。

这对于图的中间工作逻辑需要的任何内容都很有用，但与整体图输入或输出无关。

我们将定义一个`OverallState`和一个`PrivateState`。

`node_2`使用`PrivateState`作为输入，但写出到`OverallState`。

In [ ]:
from typing_extensions import TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

class OverallState(TypedDict):
    foo: int

class PrivateState(TypedDict):
    baz: int

def node_1(state: OverallState) -> PrivateState:
    print("---节点 1---")
    return {"baz": state['foo'] + 1}

def node_2(state: PrivateState) -> OverallState:
    print("---节点 2---")
    return {"foo": state['baz'] + 1}

# 构建图
builder = StateGraph(OverallState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"foo" : 1})

`baz`只包含在`PrivateState`中。

`node_2`使用`PrivateState`作为输入，但写出到`OverallState`。

所以，我们可以看到`baz`被从图输出中排除，因为它不在`OverallState`中。

## 输入/输出模式

默认情况下，`StateGraph`接受单个模式，所有节点都期望与该模式通信。

但是，也可以[为图定义显式的输入和输出模式](https://langchain-ai.github.io/langgraph/how-tos/input_output_schema/?h=input+outp)。

通常，在这些情况下，我们定义一个包含与图操作相关的*所有*键的"内部"模式。

但是，我们使用特定的`input`和`output`模式来约束输入和输出。

首先，让我们只使用单个模式运行图。

In [ ]:
class OverallState(TypedDict):
    question: str
    answer: str
    notes: str

def thinking_node(state: OverallState):
    return {"answer": "再见", "notes": "... 他的名字是Lance"}

def answer_node(state: OverallState):
    return {"answer": "再见 Lance"}

graph = StateGraph(OverallState)
graph.add_node("answer_node", answer_node)
graph.add_node("thinking_node", thinking_node)
graph.add_edge(START, "thinking_node")
graph.add_edge("thinking_node", "answer_node")
graph.add_edge("answer_node", END)

graph = graph.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

注意invoke的输出包含`OverallState`中的所有键。

In [ ]:
graph.invoke({"question":"你好"})

现在，让我们在图中使用特定的`input`和`output`模式。

在这里，`input`/`output`模式对图的输入和输出允许的键执行*过滤*。

此外，我们可以使用类型提示`state: InputState`来指定每个节点的输入模式。

当图使用多重模式时，这很重要。

我们在下面使用类型提示，例如，显示`answer_node`的输出将被过滤为`OutputState`。

In [ ]:
class InputState(TypedDict):
    question: str

class OutputState(TypedDict):
    answer: str

class OverallState(TypedDict):
    question: str
    answer: str
    notes: str

def thinking_node(state: InputState):
    return {"answer": "再见", "notes": "... 他的名字是Lance"}

def answer_node(state: OverallState) -> OutputState:
    return {"answer": "再见 Lance"}

graph = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
graph.add_node("answer_node", answer_node)
graph.add_node("thinking_node", thinking_node)
graph.add_edge(START, "thinking_node")
graph.add_edge("thinking_node", "answer_node")
graph.add_edge("answer_node", END)

graph = graph.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

graph.invoke({"question":"你好"})

我们可以看到`output`模式将输出约束为仅包含`answer`键。